# GSB솔루션 공정 품질 분석

**실행 방법: 상단 Run → Run All Cells**를 클릭하세요. 아래의 1~7단계가 순서대로 실행됩니다.

기본 입력은 `data/raw/gsb_input.xlsx`입니다. 원본을 수정하지 않고 매 실행 결과를 별도 폴더에 저장합니다.

현재 데이터에서 확인하는 것은 LOT 기록값의 관계입니다. (LOT 대표성·불량 판정유형을 확인하기 전에는 이를 개별 제품의 불량확률로 해석하지 않습니다.)

[사용 안내](docs/user-guide.md) · [설치·활용 가이드](docs/install-guide.md)

## 1. 실행 준비
입력 파일을 바꾸려면 운영자가 `INPUT_FILE`만 수정합니다. 규격은 현재 대표 품번에 적용합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML

ROOT = Path.cwd()
if not (ROOT / 'src/gsb_analysis.py').exists():
    raise RuntimeError('프로젝트 루트에서 노트북을 실행하세요.')
sys.path.insert(0, str(ROOT))
from src.gsb_analysis import load_data, summarize, quality_checks, plots, measured_flags, save_results
INPUT_FILE = ROOT / 'data/raw/gsb_input.xlsx'
print('분석 준비 완료')

## 2. 입력 및 전처리
빈칸·`-`는 결측으로 유지합니다. 숫자 오류나 중복 LOT가 있으면 중단하여 잘못된 집계를 방지합니다.

In [ ]:
data = load_data(INPUT_FILE)
if set(data['part_no']) != {'11404 10806P'}:
    raise ValueError('현재 규격은 11404 10806P용입니다. 다른 품번은 규격을 확인한 후 별도로 분석하세요.')
print(f'자료 로딩 완료: {len(data):,}개 LOT')
display(data.head(10))

## 3. 데이터 품질과 연결률
동일 LOT 행에 두 값이 존재하는 비율입니다.

In [ ]:
tables = summarize(data)
tables['quality_checks'] = quality_checks(data)
display(tables['coverage'].round(2))
display(tables['quality_checks'])

## 4. 불량수량 및 규격 비교
결측을 0으로 바꾸지 않습니다. PPM은 생산수량이 양수이고 불량수량이 기록된 동일 LOT만으로 계산합니다. 생산수량 정의는 기업 확인이 필요합니다. 기록 치수의 규격 이탈 건수와 불량품 수량은 서로 다른 지표입니다.

In [ ]:
display(tables['defect_counts'].round(2))
display(tables['specifications'])
print('규격 내 기록값이 LOT 전체 무불량을 의미하지는 않습니다.')

## 5. 피어슨 상관계수와 시각화
피어슨 r은 두 변수의 선형 관계를 −1~+1로 나타냅니다. 0에 가까우면 뚜렷한 선형 관계가 약하지만, 비선형 관계나 다른 공정변수의 영향까지 배제하지 않습니다.

산점도의 원 면적은 같은 좌표의 LOT 개수에 비례합니다. 그래프는 실제 측정 좌표를 사용합니다. 추세 그래프의 붉은 선은 작업표준서 정상 범위 규격입니다.

In [ ]:
display(tables['correlation'].round(4))
figures = plots(data)
for figure in figures.values():
    display(figure)
import matplotlib.pyplot as plt
plt.close('all')

## 6. 기록값 점검 및 분석 한계
아래 판정은 이미 기록된 값의 규격 점검입니다. (미래 불량 사전예측 기능은 아닙니다.)

In [ ]:
tables['measured_flags'] = measured_flags(data)
flags = tables['measured_flags']
mask = flags.filter(like='_status').ne('기록값 규격 내').any(axis=1)
display(flags.loc[mask].head(30))
thread_spec = tables['specifications'].query("항목 == 'thread_diameter_mm'").iloc[0]
if thread_spec['규격 이탈 수'] == 0:
    print('규격 이탈 실측 사례가 없어 불량예측과 위험 임계치 검증을 진행하지 않습니다.')
else:
    print('이탈 기록이 있습니다. 측정값 대표성·불량유형·표본수를 확인한 후 예측모델을 별도로 검증해야 합니다.')
display(Markdown('추가 필요자료: **LOT 평균/대표 샘플 여부, 불량품 실측값, 샘플·LOT·검사시간, 불량유형, 공정 조건 변경 이력**'))

## 7. 결과 저장
CSV·그림·HTML 보고서를 실행별 폴더에 저장합니다. 원본 파일 해시와 실행 환경도 기록합니다.

In [ ]:
result_dir = save_results(data, tables, figures, INPUT_FILE, ROOT)
relative = result_dir.relative_to(ROOT).as_posix()
display(Markdown(f'## 분석 완료\n[결과 폴더 열기]({relative}/) · [HTML 보고서]({relative}/report.html)'))
print(f'저장 위치: {relative}')